# 🚀 Starflow AI Lab 001：第一次自主训练视频预测模型

**只需要 iPhone + 免费 Google Colab，默认 CPU，零 API 调用。**

这是一个真正会调整神经网络权重的 PyTorch 小实验：根据两张 24×24 的模拟运动帧，预测第三帧。**它不是真人视频生成模型，也不具备 4K / 120FPS 功能。**

**iPhone 操作：**登录 Google 后点上方「运行时 → 全部运行」（或依次点每格左边 ▶）；无需设置 GPU。训练完成会显示图表并生成 `starflow_tiny_motion_v0.pt`。最后一个格子用于手动下载模型文件。

本实验不读取相册、不会上传个人视频，也不会请求收费 AI 服务。首次使用 Colab 可能要求授权连接免费运行环境；不要购买计算单元。

## 1. 生成训练数据并训练模型

训练集和测试集都是代码在云端内存中生成的模拟光点运动。模型参数通过反向传播和 Adam 优化器真正更新；不会使用第三方预训练权重。

In [ ]:
# Starflow AI Lab 001：两帧预测下一帧（原创的小型卷积网络）
# 默认 CPU；无需 GPU、API Key、付费服务或外部训练数据。
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SIZE = 24
EPOCHS = 12
torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device("cpu")

# 在内存中合成三帧连续的“运动亮点”，无需下载数据。
# 帧0、帧1作为输入，帧2作为训练标签；测试数据采用不同随机种子。
def make_data(n: int, seed: int):
    rng = torch.Generator().manual_seed(seed)
    pos = torch.randint(5, SIZE - 5, (n, 2), generator=rng).float()
    speed_options = torch.tensor([-2., -1., 1., 2.])
    vx = speed_options[torch.randint(0, 4, (n,), generator=rng)]
    vy = speed_options[torch.randint(0, 4, (n,), generator=rng)]
    yy, xx = torch.meshgrid(
        torch.arange(SIZE).float(), torch.arange(SIZE).float(), indexing="ij"
    )
    def frame(t: int):
        cx = (pos[:, 0] + vx * t)[:, None, None]
        cy = (pos[:, 1] + vy * t)[:, None, None]
        return torch.exp(-((xx - cx)**2 + (yy - cy)**2)/(2 * 1.7**2)).unsqueeze(1)
    f0, f1, f2 = frame(0), frame(1), frame(2)
    return torch.cat((f0, f1), dim=1), f2

train_x, train_y = make_data(1024, SEED)
test_x, test_y = make_data(256, 99)

# 约 9,713 个参数的原创学习实验：只学习简单平面运动，
# 不是从零训练的通用视频基础模型。
class StarflowTinyMotion(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 3, padding=1)
        )
    def forward(self, x):
        # 预测相对最近一帧的变化。
        return x[:, 1:2] + self.net(x)

model = StarflowTinyMotion().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
loader = DataLoader(
    TensorDataset(train_x, train_y), batch_size=64, shuffle=True
)
with torch.no_grad():
    baseline = ((test_x[:, 1:2] - test_y)**2).mean().item()
print(f"模型参数量：{sum(p.numel() for p in model.parameters()):,}")
print(f"基线误差（直接重复第2帧）：{baseline:.6f}")

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in loader:
        prediction = model(xb)
        # 加权损失：避免由于背景像素过多而只学会输出黑色。
        loss = (((prediction - yb)**2) * (1 + 4 * yb)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        predicted = model(test_x).clamp(0, 1)
        test_mse = ((predicted - test_y)**2).mean().item()
    history.append(test_mse)
    print(f"第 {epoch:02d}/{EPOCHS} 轮 | 测试 MSE: {test_mse:.6f}")

print("训练完成。数值只反映合成运动亮点数据集，不代表真人视频生成质量。")

## 2. 看训练结果、保存模型

查看预测的第三帧是否比直接重复第二帧更接近真实第三帧。MSE 越低越好；这是玩具实验的内部指标，不能用于宣传真实视频质量。

In [ ]:
# 对照：上一帧直接复制（基线） vs 模型预测 vs 真实下一帧
model.eval()
with torch.no_grad():
    example = 8
    predicted = model(test_x[example:example+1]).clamp(0, 1)[0,0].cpu().numpy()
    previous = test_x[example,1].cpu().numpy()
    actual = test_y[example,0].cpu().numpy()
    mse = history[-1]
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, img, title in zip(
    axes,
    [previous, predicted, actual],
    ["第 2 帧（基线）", "Starflow 预测第 3 帧", "真实第 3 帧"]
):
    ax.imshow(img, cmap="magma", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.plot(range(1, EPOCHS + 1), history, marker="o", label="模型测试误差")
plt.axhline(baseline, linestyle="--", color="gray", label="上一帧基线误差")
plt.xlabel("训练轮次")
plt.ylabel("MSE 越低越好")
plt.legend()
plt.grid(alpha=.2)
plt.show()
print(f"基线 MSE：{baseline:.6f} | 模型 MSE：{mse:.6f}")
print(f"相对基线误差下降：{100 * (baseline - mse)/baseline:.1f}%（仅此玩具数据集）")

# 将学到的权重保存为自己的实验文件；不依赖任何商业模型。
out_path = "starflow_tiny_motion_v0.pt"
torch.save({
    "model_state_dict": model.cpu().state_dict(),
    "size": SIZE, "epochs": EPOCHS, "seed": SEED,
    "test_mse": mse, "baseline_mse": baseline,
    "description": "Starflow AI Lab 001 synthetic moving-dot prediction"
}, out_path)
print(f"已保存权重：{out_path}。在 Colab 左侧“文件”中可找到并下载。")

## 3. （可选）下载你的第一个模型权重

如果 iPhone Safari 的文件下载被阻止，可使用 Colab 左侧的「文件」面板下载 `starflow_tiny_motion_v0.pt`。如果运行环境重置，需要重新训练并保存。

In [ ]:
# 可选：要把训练出的模型文件下载到 iPhone 时，单独运行本格。
# 如果不想下载，跳过即可。
from google.colab import files
files.download("starflow_tiny_motion_v0.pt")